In [ ]:
from easyAI import TwoPlayerGame, Human_Player, AI_Player, Negamax

class GameOfBones( TwoPlayerGame ):
    """ In turn, the players remove one, two or three bones from a
    pile of bones. The player who removes the last bone loses. """

    def __init__(self, players=None):
        self.players = players
        self.pile = 20 # start with 20 bones in the pile
        self.current_player = 1 # player 1 starts

    def possible_moves(self): return ['1','2','3']
    def make_move(self,move): self.pile -= int(move) # remove bones.
    def win(self): return self.pile<=0 # opponent took the last bone ?
    def is_over(self): return self.win() # Game stops when someone wins.
    def show(self): print ("%d bones left in the pile" % self.pile)
    def scoring(self): return 100 if game.win() else 0 # For the AI

# Start a match (and store the history of moves when it ends)
ai = Negamax(13) # The AI will think 13 moves in advance
game = GameOfBones( [ Human_Player(), AI_Player(ai) ] )
history = game.play()

20 bones left in the pile

Move #1: player 1 plays 3 :
17 bones left in the pile

Move #2: player 2 plays 1 :
16 bones left in the pile

Move #3: player 1 plays 2 :
14 bones left in the pile

Move #4: player 2 plays 1 :
13 bones left in the pile

Move #5: player 1 plays 3 :
10 bones left in the pile

Move #6: player 2 plays 1 :
9 bones left in the pile

Move #7: player 1 plays 2 :
7 bones left in the pile

Move #8: player 2 plays 1 :
6 bones left in the pile

Move #9: player 1 plays 2 :
4 bones left in the pile

Move #10: player 2 plays 1 :
3 bones left in the pile

Move #11: player 1 plays 2 :
1 bones left in the pile

Move #12: player 2 plays 1 :
0 bones left in the pile


In [3]:
try:
    import numpy as np
except ImportError:
    print("Sorry, this example requires Numpy installed !")
    raise

from easyAI import TwoPlayerGame, AI_Player, Negamax


class ConnectFour(TwoPlayerGame):
    """
    Connect Four with optional move noise.

    If noise_prob > 0, the chosen column can be randomly shifted
    by one to the left/right before placing the token.
    """

    def __init__(self, players, board=None, noise_prob=0.0, seed=None):
        self.players = players
        self.board = board if (board is not None) else np.zeros((6, 7), dtype=int)
        self.current_player = 1
        self.noise_prob = float(noise_prob)
        self.rng = np.random.default_rng(seed)

    def possible_moves(self):
        return [i for i in range(7) if self.board[:, i].min() == 0]

    def randomly_change_col(self, column):
        if self.noise_prob <= 0.0:
            return int(column)

        if self.rng.random() >= self.noise_prob:
            return int(column)

        candidates = []
        if column > 0:
            candidates.append(column - 1)
        if column < 6:
            candidates.append(column + 1)

        if not candidates:
            return int(column)

        return int(self.rng.choice(candidates))

    def make_move(self, column):
        base_column = int(column)
        column = self.randomly_change_col(base_column)

        legal = self.possible_moves()
        if column not in legal:
            column = base_column if base_column in legal else legal[0]

        line = int(np.where(self.board[:, column] == 0)[0][0])
        self.board[line, column] = self.current_player

    def show(self):
        print(
            "\n"
            + "\n".join(
                ["0 1 2 3 4 5 6", 13 * "-"]
                + [
                    " ".join([".", "O", "X"][self.board[5 - j][i]] for i in range(7))
                    for j in range(6)
                ]
            )
        )

    def lose(self):
        return find_four(self.board, self.opponent_index)

    def is_over(self):
        return (self.board.min() > 0) or self.lose()

    def scoring(self):
        return -100 if self.lose() else 0


def find_four(board, player_id):
    """Returns True iff given player has connected 4 or more."""
    for pos, direction in POS_DIR:
        streak = 0
        while (0 <= pos[0] <= 5) and (0 <= pos[1] <= 6):
            if board[pos[0], pos[1]] == player_id:
                streak += 1
                if streak == 4:
                    return True
            else:
                streak = 0
            pos = pos + direction
    return False


POS_DIR = np.array(
    [[[i, 0], [0, 1]] for i in range(6)]
    + [[[0, i], [1, 0]] for i in range(7)]
    + [[[i, 0], [1, 1]] for i in range(1, 3)]
    + [[[0, i], [1, 1]] for i in range(4)]
    + [[[i, 6], [1, -1]] for i in range(1, 3)]
    + [[[0, i], [1, -1]] for i in range(3, 7)]
)


def play_ai_game(depth_a, depth_b, starter_agent="A", noise_prob=0.0, seed=None, verbose=False):
    """
    Plays one game between two Negamax agents.

    Agent A uses depth_a, Agent B uses depth_b.
    starter_agent: "A" or "B".
    """
    player_a = AI_Player(Negamax(depth_a))
    player_b = AI_Player(Negamax(depth_b))

    if starter_agent == "A":
        players = [player_a, player_b]
        slot_to_agent = {1: "A", 2: "B"}
    else:
        players = [player_b, player_a]
        slot_to_agent = {1: "B", 2: "A"}

    game = ConnectFour(players, noise_prob=noise_prob, seed=seed)
    history = game.play(verbose=verbose)

    winner_slot = game.opponent_index if game.lose() else None
    winner_agent = slot_to_agent[winner_slot] if winner_slot is not None else None

    return {
        "winner_agent": winner_agent,
        "winner_slot": winner_slot,
        "starter_agent": starter_agent,
        "moves": max(0, len(history) - 1),
    }


def benchmark_negamax(
    depth_pairs=((2, 7), (1, 4)),
    games_per_pair=10,
    noise_levels=(0.0, 0.5),
    base_seed=1234,
):
    """
    Runs A-vs-B Negamax benchmarks for multiple depths and game variants.

    - Alternates starting player every game.
    - Compares deterministic (noise=0.0) and probabilistic variants.
    """
    rng = np.random.default_rng(base_seed)
    results = []
    game_count = 0
    max_games = len(depth_pairs) * len(noise_levels) * games_per_pair

    for noise_prob in noise_levels:
        variant = "deterministic" if noise_prob == 0.0 else f"probabilistic(p={noise_prob:.2f})"

        for depth_a, depth_b in depth_pairs:
            stats = {
                "variant": variant,
                "depth_a": depth_a,
                "depth_b": depth_b,
                "games": games_per_pair,
                "a_wins": 0,
                "b_wins": 0,
                "draws": 0,
                "slot1_wins": 0,
                "slot2_wins": 0,
                "a_wins_as_starter": 0,
                "a_wins_as_second": 0,
                "b_wins_as_starter": 0,
                "b_wins_as_second": 0,
                "total_moves": 0,
            }

            for game_idx in range(games_per_pair):
                game_count += 1
                starter_agent = "A" if game_idx % 2 == 0 else "B"
                game_seed = int(rng.integers(0, 2**32 - 1))

                outcome = play_ai_game(
                    depth_a=depth_a,
                    depth_b=depth_b,
                    starter_agent=starter_agent,
                    noise_prob=noise_prob,
                    seed=game_seed,
                    verbose=False,
                )

                stats["total_moves"] += outcome["moves"]

                if outcome["winner_slot"] == 1:
                    stats["slot1_wins"] += 1
                elif outcome["winner_slot"] == 2:
                    stats["slot2_wins"] += 1

                if outcome["winner_agent"] == "A":
                    stats["a_wins"] += 1
                    if outcome["starter_agent"] == "A":
                        stats["a_wins_as_starter"] += 1
                    else:
                        stats["a_wins_as_second"] += 1
                elif outcome["winner_agent"] == "B":
                    stats["b_wins"] += 1
                    if outcome["starter_agent"] == "B":
                        stats["b_wins_as_starter"] += 1
                    else:
                        stats["b_wins_as_second"] += 1
                else:
                    stats["draws"] += 1

                print(f"Played {game_count}/{max_games}")

            g = stats["games"]
            stats["a_win_rate"] = stats["a_wins"] / g
            stats["b_win_rate"] = stats["b_wins"] / g
            stats["draw_rate"] = stats["draws"] / g
            stats["avg_moves"] = stats["total_moves"] / g


            results.append(stats)

    return results


def print_benchmark(results):
    header = (
        f"{'variant':24} {'depth(A:B)':10} {'A_wins':>7} {'B_wins':>7} "
        f"{'draws':>7} {'A_rate':>8} {'B_rate':>8} {'draw_rate':>10} {'avg_moves':>10}"
    )
    print(header)
    print("-" * len(header))

    for r in results:
        print(
            f"{r['variant']:24} {r['depth_a']}:{r['depth_b']:<7} "
            f"{r['a_wins']:7d} {r['b_wins']:7d} {r['draws']:7d} "
            f"{r['a_win_rate']:8.2%} {r['b_win_rate']:8.2%} {r['draw_rate']:10.2%} {r['avg_moves']:10.2f}"
        )


results = benchmark_negamax(
    depth_pairs=[(1,4), (2, 7)],
    games_per_pair=10,
    noise_levels=[0.0, 0.5],
    base_seed=42,
)
print_benchmark(results)
results




Played 1/40
Played 2/40
Played 3/40
Played 4/40
Played 5/40
Played 6/40
Played 7/40
Played 8/40
Played 9/40
Played 10/40
Played 11/40
Played 12/40
Played 13/40
Played 14/40
Played 15/40
Played 16/40
Played 17/40
Played 18/40
Played 19/40
Played 20/40
Played 21/40
Played 22/40
Played 23/40
Played 24/40
Played 25/40
Played 26/40
Played 27/40
Played 28/40
Played 29/40
Played 30/40
Played 31/40
Played 32/40
Played 33/40
Played 34/40
Played 35/40
Played 36/40
Played 37/40
Played 38/40
Played 39/40
Played 40/40
variant                  depth(A:B)  A_wins  B_wins   draws   A_rate   B_rate  draw_rate  avg_moves
---------------------------------------------------------------------------------------------------
deterministic            1:4             0      10       0    0.00%  100.00%      0.00%      18.50
deterministic            2:7             5       5       0   50.00%   50.00%      0.00%      38.00
probabilistic(p=0.50)    1:4             1       9       0   10.00%   90.00%      0.00%    

[{'variant': 'deterministic',
  'depth_a': 1,
  'depth_b': 4,
  'games': 10,
  'a_wins': 0,
  'b_wins': 10,
  'draws': 0,
  'slot1_wins': 5,
  'slot2_wins': 5,
  'a_wins_as_starter': 0,
  'a_wins_as_second': 0,
  'b_wins_as_starter': 5,
  'b_wins_as_second': 5,
  'total_moves': 185,
  'a_win_rate': 0.0,
  'b_win_rate': 1.0,
  'draw_rate': 0.0,
  'avg_moves': 18.5},
 {'variant': 'deterministic',
  'depth_a': 2,
  'depth_b': 7,
  'games': 10,
  'a_wins': 5,
  'b_wins': 5,
  'draws': 0,
  'slot1_wins': 0,
  'slot2_wins': 10,
  'a_wins_as_starter': 0,
  'a_wins_as_second': 5,
  'b_wins_as_starter': 0,
  'b_wins_as_second': 5,
  'total_moves': 380,
  'a_win_rate': 0.5,
  'b_win_rate': 0.5,
  'draw_rate': 0.0,
  'avg_moves': 38.0},
 {'variant': 'probabilistic(p=0.50)',
  'depth_a': 1,
  'depth_b': 4,
  'games': 10,
  'a_wins': 1,
  'b_wins': 9,
  'draws': 0,
  'slot1_wins': 6,
  'slot2_wins': 4,
  'a_wins_as_starter': 1,
  'a_wins_as_second': 0,
  'b_wins_as_starter': 5,
  'b_wins_as_second'

In [4]:
# 2. Porównaj kilka (co najmniej Negamax z i bez odcięcia alfa-beta, z
# dwoma różnymi ustawieniami maksymalnej głębokości) algorytmy na
# deterministycznych i probabilistycznych wariantach twojej gry.
# 3. Napisz kod, który mierzy średni czas spędzony na wybieraniu akcji
# przez każdy wariant AI. Dodaj zmierzone czasy do raportu.
import time
from math import inf
from dataclasses import dataclass

import numpy as np
from easyAI import AI_Player, Negamax

#Negamax bez alfa-beta
def negamax_no_ab(game, depth, orig_depth, scoring):
    if (depth == 0) or game.is_over():
        return scoring(game) * (1 + 0.001 * depth)

    state = game
    moves = game.possible_moves()
    best_val = -inf

    if depth == orig_depth:
        state.ai_move = moves[0]

    unmake = hasattr(state, "unmake_move")

    for move in moves:
        if not unmake:
            game = state.copy()

        game.make_move(move)
        game.switch_player()

        val = -negamax_no_ab(game, depth - 1, orig_depth, scoring)

        if unmake:
            game.switch_player()
            game.unmake_move(move)

        if val > best_val:
            best_val = val
            if depth == orig_depth:
                state.ai_move = move

    return best_val


class NegamaxNoAB:
    def __init__(self, depth, scoring=None):
        self.depth = depth
        self.scoring = scoring
        self.alpha = None

    def __call__(self, game):
        scoring = self.scoring if self.scoring else (lambda g: g.scoring())
        self.alpha = negamax_no_ab(game, self.depth, self.depth, scoring)
        return game.ai_move


#Player z pomiarem czasu
class TimedAIPlayer(AI_Player):
    def __init__(self, AI_algo, name="AI"):
        super().__init__(AI_algo, name=name)
        self.total_decision_time = 0.0
        self.decision_count = 0

    def ask_move(self, game):
        t0 = time.perf_counter()
        move = super().ask_move(game)
        self.total_decision_time += (time.perf_counter() - t0)
        self.decision_count += 1
        return move


@dataclass(frozen=True)
class Variant:
    name: str
    algo: str   # "negamax_ab" | "negamax_no_ab"
    depth: int


def make_algo(variant):
    if variant.algo == "negamax_ab":
        return Negamax(variant.depth)
    if variant.algo == "negamax_no_ab":
        return NegamaxNoAB(variant.depth)
    raise ValueError(f"Unknown algo: {variant.algo}")


def play_ai_game_variant(variant_a, variant_b, starter_agent="A", noise_prob=0.0, seed=None, verbose=False):
    player_a = TimedAIPlayer(make_algo(variant_a), name=f"A:{variant_a.name}")
    player_b = TimedAIPlayer(make_algo(variant_b), name=f"B:{variant_b.name}")

    if starter_agent == "A":
        players = [player_a, player_b]
        slot_to_agent = {1: "A", 2: "B"}
    else:
        players = [player_b, player_a]
        slot_to_agent = {1: "B", 2: "A"}

    game = ConnectFour(players, noise_prob=noise_prob, seed=seed)
    history = game.play(verbose=verbose)

    winner_slot = game.opponent_index if game.lose() else None
    winner_agent = slot_to_agent[winner_slot] if winner_slot is not None else None

    return {
        "winner_agent": winner_agent,
        "moves": max(0, len(history) - 1),
        "a_time_total": player_a.total_decision_time,
        "b_time_total": player_b.total_decision_time,
        "a_decisions": player_a.decision_count,
        "b_decisions": player_b.decision_count,
    }


def benchmark_variants(variant_pairs, games_per_pair=50, noise_levels=(0.0, 0.2), base_seed=42):
    rng = np.random.default_rng(base_seed)
    results = []
    total_games = len(variant_pairs) * len(noise_levels) * games_per_pair
    counter = 0

    for noise_prob in noise_levels:
        env_variant = "deterministic" if noise_prob == 0.0 else f"probabilistic(p={noise_prob:.2f})"

        for variant_a, variant_b in variant_pairs:
            stats = {
                "env_variant": env_variant,
                "A_variant": variant_a.name,
                "B_variant": variant_b.name,
                "games": games_per_pair,
                "A_wins": 0,
                "B_wins": 0,
                "draws": 0,
                "total_moves": 0,
                "A_time_total": 0.0,
                "B_time_total": 0.0,
                "A_decisions": 0,
                "B_decisions": 0,
            }

            for i in range(games_per_pair):
                counter += 1
                starter = "A" if i % 2 == 0 else "B"
                seed = int(rng.integers(0, 2**32 - 1))

                out = play_ai_game_variant(
                    variant_a=variant_a,
                    variant_b=variant_b,
                    starter_agent=starter,
                    noise_prob=noise_prob,
                    seed=seed,
                    verbose=False,
                )

                stats["total_moves"] += out["moves"]
                stats["A_time_total"] += out["a_time_total"]
                stats["B_time_total"] += out["b_time_total"]
                stats["A_decisions"] += out["a_decisions"]
                stats["B_decisions"] += out["b_decisions"]

                if out["winner_agent"] == "A":
                    stats["A_wins"] += 1
                elif out["winner_agent"] == "B":
                    stats["B_wins"] += 1
                else:
                    stats["draws"] += 1

                print(f"Played {counter}/{total_games}", end="\r")

            g = stats["games"]
            stats["A_rate"] = stats["A_wins"] / g
            stats["B_rate"] = stats["B_wins"] / g
            stats["draw_rate"] = stats["draws"] / g
            stats["avg_moves"] = stats["total_moves"] / g
            stats["A_avg_decision_ms"] = 1000.0 * stats["A_time_total"] / max(1, stats["A_decisions"])
            stats["B_avg_decision_ms"] = 1000.0 * stats["B_time_total"] / max(1, stats["B_decisions"])

            results.append(stats)

    print()
    return results


def print_benchmark_with_time(results):
    header = (
        f"{'env_variant':24} {'A_vs_B':34} {'A_wins':>7} {'B_wins':>7} {'draws':>7} "
        f"{'A_rate':>8} {'B_rate':>8} {'draw_rate':>10} {'avg_moves':>10} {'A_ms':>10} {'B_ms':>10}"
    )
    print(header)
    print("-" * len(header))

    for r in results:
        pair = f"{r['A_variant']} vs {r['B_variant']}"
        print(
            f"{r['env_variant']:24} {pair:34} "
            f"{r['A_wins']:7d} {r['B_wins']:7d} {r['draws']:7d} "
            f"{r['A_rate']:8.2%} {r['B_rate']:8.2%} {r['draw_rate']:10.2%} {r['avg_moves']:10.2f} "
            f"{r['A_avg_decision_ms']:10.2f} {r['B_avg_decision_ms']:10.2f}"
        )


def summarize_time_by_variant(results):
    agg = {}
    for r in results:
        for side in ("A", "B"):
            name = r[f"{side}_variant"]
            agg.setdefault(name, {"time": 0.0, "decisions": 0})
            agg[name]["time"] += r[f"{side}_time_total"]
            agg[name]["decisions"] += r[f"{side}_decisions"]

    rows = []
    for name, d in agg.items():
        rows.append({
            "variant": name,
            "decisions": d["decisions"],
            "avg_decision_ms": 1000.0 * d["time"] / max(1, d["decisions"]),
        })
    rows.sort(key=lambda x: x["variant"])
    return rows


def print_time_summary(rows):
    header = f"{'variant':26} {'decisions':>10} {'avg_decision_ms':>18}"
    print(header)
    print("-" * len(header))
    for r in rows:
        print(f"{r['variant']:26} {r['decisions']:10d} {r['avg_decision_ms']:18.2f}")



In [ ]:
v_noab_d3 = Variant("negamax_no_ab_d3", "negamax_no_ab", 3)
v_ab_d3   = Variant("negamax_ab_d3", "negamax_ab", 3)
v_noab_d4 = Variant("negamax_no_ab_d4", "negamax_no_ab", 4)
v_ab_d4   = Variant("negamax_ab_d4", "negamax_ab", 4)

variant_pairs = [
    (v_noab_d3, v_ab_d3),
    (v_noab_d4, v_ab_d4),
]

results = benchmark_variants(
    variant_pairs=variant_pairs,
    games_per_pair=10,
    noise_levels=(0.0, 0.2),
    base_seed=42,
)

print_benchmark_with_time(results)

print("\n=== Średni czas wyboru akcji (po wszystkich meczach) ===")
time_rows = summarize_time_by_variant(results)
print_time_summary(time_rows)

results, time_rows
